In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

cols = ["unit", "cycle", "op1", "op2","op3","s1", "s2", "s3", "s4", "s5", "s6", "s7", "s8", "s9", "s10", "s11", "s12", "s13", "s14", "s15", "s16", "s17", "s18", "s19", "s20", "s21"]

train = pd.read_csv("../data/raw/train_FD001.txt", sep = r"\s+", header = None, names = cols)
test = pd.read_csv("../data/raw/test_FD001.txt", sep = r"\s+", header = None, names = cols)
rul = pd.read_csv("../data/raw/RUL_FD001.txt", sep = r"\s+", header = None, names = ["RUL"])
rul.index = range(1,len(rul)+1)

aus eda_01:

In [2]:
drop_cols = ["op1", "op2","op3", "s1", "s5", "s6", "s10", "s16", "s18", "s19"]

In [3]:
train["max_cycle"] = train.groupby("unit")["cycle"].transform("max")
train["RUL"] = train["max_cycle"] - train["cycle"]

In [4]:
up = ["s2", "s3", "s4", "s8", "s11", "s13", "s15", "s17"]
down= ["s7", "s12", "s20", "s21"]
ambiguous=["s9", "s14"]

features = up + down + ambiguous
len(features)

14

In [5]:
train["RUL_clip"] = train["RUL"].clip(upper=125)
X_train = train[features]
y_train = train["RUL_clip"]
X_test = test.groupby("unit")[features].last()
y_test = rul["RUL"]

In [6]:
X_test.shape

(100, 14)

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [8]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

np.sqrt(mean_squared_error(y_test, y_pred))

np.float64(21.900213406890536)

In [9]:
from xgboost import XGBRegressor
xgb_model = XGBRegressor(random_state=42)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

In [10]:
np.sqrt(mean_squared_error(y_test, y_pred_xgb))

np.float64(18.177173527563955)

In [17]:
y_pred_train_xgb = xgb_model.predict(X_train)
print(y_pred_train_xgb)
np.sqrt(mean_squared_error(y_train, y_pred_train_xgb))

[119.55811   118.14523   121.27614   ...   5.387664    1.0365708
   2.2939532]


np.float64(12.391884738531378)

Anzeichen für Overfitting da RMSE der Testdaten bei 12,4 und bei den Testdaten bei 18,2 also ca 50% höher. Aufgrund unteschiedlicher Messdaten allerdings nicht vergleichbar 

In [12]:
y_pred_train = model.predict(X_train)
np.sqrt(mean_squared_error(y_train, y_pred_train))

np.float64(21.491018701515173)

In [13]:
y_pred_train.min()

np.float64(-79.75576181896031)

In [14]:
y_pred_train.max()

np.float64(159.2387168622663)

In [18]:
print(f"Testtriebwerke mit RUL > 125: {(rul['RUL']>125).sum()}")
print(f"XGBoost Train: min {y_pred_train_xgb.min():.1f}, max {y_pred_train_xgb.max():.1f}")

Testtriebwerke mit RUL > 125: 11
XGBoost Train: min -2.2, max 134.5


In [19]:
print(f"XGBoost Train RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train_xgb)):.2f}")
print(f"LinReg Vorhersagen <0: {(y_pred_train < 0).sum()}")

XGBoost Train RMSE: 12.39
LinReg Vorhersagen <0: 465


## Ergebnisse der Baseline

| Modell | RMSE Train (alle Zyklen) | RMSE Test (letzter Zyklus, 100 Triebwerke) |
|---|---|---|
| Lineare Regression | 21,49 | 21,9 |
| XGBoost (Standardparameter) | 12,39 | 18,2 |

**Einordnung**
- **LinReg underfittet:** Train- und Test-RMSE liegen fast gleichauf. Das Modell hat sich nichts gemerkt, ist aber zu einfach. Eine lineare Funktion kann weder das Plateau bei 125 noch das Abknicken der Sensoren vor dem Ausfall abbilden.
- **Unbegrenzte Vorhersagen:** Auf Train sagt die LinReg Werte von −79 bis 159 voraus. 465 Zeilen erhalten eine physikalisch unmögliche negative Restlebensdauer, weil eine Ebene nach oben und unten nicht begrenzt ist.
- **XGBoost** senkt den Test-RMSE auf 18,2, das sind rund 17 % weniger. Seine Vorhersagen bleiben nahezu im Labelbereich (−2,2 bis 134,5). Die Blätter der Bäume enthalten Mittelwerte von Trainingslabels. Leichte Überschreitungen entstehen, weil Boosting die Korrekturen vieler Bäume addiert.
- **Kappung im Test:** 11 von 100 Testtriebwerken haben eine wahre RUL über 125. Ein auf das gekappte Label trainiertes Modell unterschätzt sie systematisch. Der Fehler geht in die sichere Richtung: Wartung zu früh statt zu spät.
- **Grenzen:** Der Test-RMSE beruht auf nur 100 Punkten und einem einzigen Split. Den belastbaren Vergleich und den Overfitting-Check für XGBoost liefert `03_validation` mit GroupKFold.